In [14]:
from pathlib import Path
import sqlite3
import pandas as pd

# Funziona se il notebook è eseguito dalla cartella notebooks/
# oppure dalla cartella principale del progetto.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "data" / "database" / "inflation.sqlite"

if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

conn = sqlite3.connect(DB_PATH)

In [15]:
pd.read_sql_query("""
    SELECT series_id, series_key, unit, geography, description, adjustment
    FROM series
    ORDER BY series_id;
""", conn)

,series_id,series_key,unit,geography,description,adjustment
0,euribor_3m,FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA,Percent per annum,Euro area monetary market,"Euribor a tre mesi, media mensile",Monthly average
1,hicp_energy,HICP.M.U2.N.NRGY00.4D0.INX,"Index, 2025=100","Euro area, changing composition","HICP energia, indice",Neither seasonally nor working day adjusted
2,hicp_total,HICP.M.U2.N.000000.4D0.INX,"Index, 2025=100","Euro area, changing composition","HICP complessivo, indice",Neither seasonally nor working day adjusted
3,industrial_production,STBS.M.I9.Y.PROD.NS0020.4D0.N.IX,Index,"Euro area 20, fixed composition","Produzione industriale, escluse costruzioni",Working day and seasonally adjusted


In [16]:
pd.read_sql_query("""
    SELECT
        series_id,
        COUNT(*) AS n_rows,
        COUNT(value) AS n_values,
        COUNT(*) - COUNT(value) AS n_missing,
        MIN(period) AS first_month,
        MAX(period) AS last_month
    FROM observations
    GROUP BY series_id
    ORDER BY series_id;
""", conn)

,series_id,n_rows,n_values,n_missing,first_month,last_month
0,euribor_3m,312,312,0,2000-01,2025-12
1,hicp_energy,312,312,0,2000-01,2025-12
2,hicp_total,312,312,0,2000-01,2025-12
3,industrial_production,312,312,0,2000-01,2025-12


In [17]:
pd.read_sql_query("""
    SELECT
        period,
        COUNT(DISTINCT series_id) AS n_series
    FROM observations
    GROUP BY period
    HAVING COUNT(DISTINCT series_id) <> 4
    ORDER BY period;
""", conn)

,period,n_series


In [18]:
pd.read_sql_query("""
    SELECT
        o.period,
        s.series_id,
        s.unit,
        o.value
    FROM observations AS o
    JOIN series AS s
        ON o.series_id = s.series_id
    WHERE o.period = '2025-12'
    ORDER BY s.series_id;
""", conn)

,period,series_id,unit,value
0,2025-12,euribor_3m,Percent per annum,2.048429
1,2025-12,hicp_energy,"Index, 2025=100",98.780000
2,2025-12,hicp_total,"Index, 2025=100",100.630000
3,2025-12,industrial_production,Index,98.500000


In [19]:
conn.close()